# Bloomberg BQNT｜股票量化分析训练
## 动量 × 低波动：从 BQL 数据到月度组合

**交付日期：2026-09-27。样例研究区间：2021-01-01 至 2025-12-31，不是最新行情。**

本 Notebook 自带全部分析函数，无需同时导入配套 `.py` 文件。按顺序运行即可；
Bloomberg 数据查询须在有相应权限的 BQuant Python 环境中执行。

**验证范围：**纯 Python 因子、回测、成本与 IC 逻辑已用模拟数据测试；
未连接用户的 Bloomberg 账户，BQL 字段、参数、返回结构和权限仍须现场验证。
本文件不含 Bloomberg 历史行情，不会自动退回模拟数据，也没有预生成的投资业绩。

**研究边界：**固定的 12 只美股仅为教学池，有事后选样/生存偏差；`PX_LAST` 的复权与分红口径尚未确认。
因此默认图表始终带有 `RETURN BASIS UNVERIFIED` 标识。没有行业/市值中性化、风险模型或实盘下单。

## 1｜研究设计与参数

| 项目 | 本模板设定 |
|---|---|
| 动量 | $P_{t-21}/P_{t-252}-1$，用观测日近似 12–1 月，不是精确自然月复制 |
| 低波动 | 63 日简单收益标准差，乘以 $\sqrt{252}$ 年化，越低越优 |
| 合成分数 | 动量百分位 × 50% + 低波动百分位 × 50% |
| 组合 | 排名前 4，等权多头；至少需要 8 只有效股票 |
| 信号与成交 | 月末收盘评分，下一个基准有观测值日期的收盘成交 |
| 持有期 | 非调仓日权重自然漂移，不做隐含的每日等权再平衡 |
| 成本 | 买、卖名义金额分别收取 10 bps；现金利息与 Sharpe 无风险利率均假设为零 |
| 比较序列 | SPY 的同字段序列，不扣比较基准交易成本；不是自动视作总回报基准 |

“跳过最近一个月”的设计与常见动量研究有关；这里采用简单教学近似，
不是 Kenneth French 的完整因子复制。见末尾来源。默认设置不构成参数最优性判断。

In [ ]:
# -*- coding: utf-8 -*-
"""BQNT 股票因子研究教学模板：动量、低波动、月度组合、成本与 Rank IC。

默认只使用 2021-01-01 至 2025-12-31 的固定演示股票池，不代表当前成分股。
Bloomberg 查询必须在有相应权限的 BQuant Python 环境中执行。
本文件没有预装、缓存或伪造 Bloomberg 数据，也不会在查询失败时改用模拟数据。
PX_LAST 的分红/拆股调整及可用字段参数须由培训师在当前环境确认。
本模板不是无生存偏差回测，也不包含下单或实盘连接。
"""

from dataclasses import dataclass
from typing import Dict, Tuple
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class Config:
    tickers: Tuple[str, ...] = (
        "AAPL US Equity", "MSFT US Equity", "AMZN US Equity",
        "GOOGL US Equity", "JPM US Equity", "BAC US Equity",
        "XOM US Equity", "CVX US Equity", "JNJ US Equity",
        "PG US Equity", "KO US Equity", "WMT US Equity",
    )
    benchmark: str = "SPY US Equity"
    start: str = "2021-01-01"
    end: str = "2025-12-31"  # 固定历史样例，不是“最新数据”
    momentum_lookback: int = 252
    momentum_skip: int = 21
    volatility_window: int = 63
    trading_days: int = 252
    momentum_weight: float = 0.5
    top_n: int = 4
    min_names: int = 8
    cost_bps: float = 10.0  # 教学假设：每一单位买入或卖出名义金额的成本
    risk_free_annual: float = 0.0  # Sharpe 的明确假设，不是当前无风险利率
    series_label: str = "PX_LAST DEMO - RETURN BASIS UNVERIFIED"

    def __post_init__(self):
        if len(set(self.tickers)) != len(self.tickers):
            raise ValueError("tickers 不能重复。")
        if self.benchmark in self.tickers:
            raise ValueError("本例将比较基准与选股池分开。")
        if not 1 <= self.top_n <= self.min_names <= len(self.tickers):
            raise ValueError("要求 1 <= top_n <= min_names <= 股票数。")
        if not 0 < self.momentum_skip < self.momentum_lookback:
            raise ValueError("要求 0 < momentum_skip < momentum_lookback。")
        if self.volatility_window < 2 or self.trading_days <= 0:
            raise ValueError("volatility_window 至少为 2，trading_days 必须为正。")
        if not 0 <= self.momentum_weight <= 1:
            raise ValueError("momentum_weight 应在 [0, 1] 内。")
        if not np.isfinite(self.cost_bps) or not 0 <= self.cost_bps < 10000:
            raise ValueError("cost_bps 必须有限，且在 [0, 10000) 内。")
        if not np.isfinite(self.risk_free_annual) or self.risk_free_annual <= -1:
            raise ValueError("risk_free_annual 必须有限且大于 -1。")
        if pd.Timestamp(self.start) >= pd.Timestamp(self.end):
            raise ValueError("start 必须早于 end。")


CFG = Config()

In [ ]:
from IPython.display import display
print(CFG)
print("本 Notebook 不自动安装库，也不自动导出 Bloomberg 数据。")

## 2｜提取与检查 Bloomberg 数据

查询层采用 `Service → Request → execute`。先检查 `raw.head()` 和列名，确认 `ID / DATE / PX`。
日频示例按比较基准有观测值的日期对齐；这不是跨市场的完整交易日历。
拒绝重复 ID/DATE，不对其取平均；不前向或后向填充个股价格。

**培训现场核查：**`px_last` 的字段定义，`dates` 与 `fill="NA"` 是否适用于当前环境，
分红/拆股/其他公司行动调整，报价币种、停牌日是否仍有重复报价，以及数据权限。
可用经核实的全收益指数替换该数据项，但证券与比较基准必须使用一致口径。

In [ ]:
def fetch_bloomberg_prices(cfg: Config) -> pd.DataFrame:
    """返回包含 ID、DATE、PX 的长表；不静默忽略接口或权限错误。"""
    try:
        import bql
    except ImportError as exc:
        raise RuntimeError(
            "未找到 Bloomberg bql。请在具有数据权限的 BQuant 环境运行；"
            "不要安装名称相同但来源不明的 PyPI 包。"
        ) from exc
    if not all(hasattr(bql, attr) for attr in ("Service", "Request")):
        raise RuntimeError("导入的 bql 不具有所需接口，请核实 BQuant 内核。")

    try:
        bq = bql.Service()
        # 不假定 PX_LAST 含分红，不猜测复权参数；由培训师确认字段口径。
        # fill='NA' 保留缺失；不使用 fill='prev' 伪造停牌日收益。
        item = bq.data.px_last(
            dates=bq.func.range(cfg.start, cfg.end), fill="NA"
        )
        request = bql.Request(
            list(cfg.tickers) + [cfg.benchmark], {"PX": item}
        )
        response = bq.execute(request)
        raw = response[0].df().reset_index()
    except Exception as exc:
        raise RuntimeError(
            "Bloomberg 请求失败。请核查 bql 版本、PX_LAST 的 dates/fill 参数、"
            "证券代码与数据权限。原始错误保留在异常链中。"
        ) from exc
    return raw


def prepare_price_panel(raw: pd.DataFrame, cfg: Config):
    """校验返回结构；按比较基准有观测值的日期对齐，不填充个股缺失。"""
    df = raw.copy()
    df.columns = [str(c).upper() for c in df.columns]
    required = {"ID", "DATE", "PX"}
    if not required.issubset(df.columns):
        raise ValueError(
            "需要 ID、DATE、PX；实际列为 %s。请先检查 response[0].df()。"
            % list(df.columns)
        )
    if df["ID"].isna().any():
        raise ValueError("返回了空证券标识。")
    df["ID"] = df["ID"].astype(str)
    # 本例为日频；统一为无时区的日期，拒绝日期不可解析的记录。
    df["DATE"] = pd.to_datetime(df["DATE"], errors="raise", utc=True)
    df["DATE"] = df["DATE"].dt.tz_convert(None).dt.normalize()
    if df["DATE"].isna().any():
        raise ValueError("DATE 包含缺失。")
    df["PX"] = pd.to_numeric(df["PX"], errors="raise")
    if np.isinf(df["PX"].to_numpy(dtype=float)).any():
        raise ValueError("价格包含无穷值。")
    if (df["PX"].dropna() <= 0).any():
        raise ValueError("本例只支持正价格/正收益指数；请检查非正值。")
    if df.duplicated(["ID", "DATE"]).any():
        raise ValueError("同一 ID/DATE 返回多行；请核查维度，不要直接取平均。")

    wide = df.pivot(index="DATE", columns="ID", values="PX").sort_index()
    expected = list(cfg.tickers) + [cfg.benchmark]
    absent = sorted(set(expected).difference(wide.columns))
    if absent:
        raise ValueError("缺少证券 %s；核查返回 ID 和原始证券代码。" % absent)
    wide = wide.loc[pd.Timestamp(cfg.start):pd.Timestamp(cfg.end), expected]
    dates = wide[cfg.benchmark].dropna().index
    if len(dates) < max(cfg.momentum_lookback, cfg.volatility_window) + 3:
        raise ValueError("有效日期不足；需要更长历史区间。")
    stock_px = wide.reindex(dates)[list(cfg.tickers)].astype(float)
    benchmark_px = wide.reindex(dates)[cfg.benchmark].astype(float)
    # 该表仅诊断，不用全样本缺失率来筛选历史股票池。
    coverage = pd.DataFrame({
        "observations": stock_px.notna().sum(),
        "missing_fraction": stock_px.isna().mean(),
        "first_observation": stock_px.apply(lambda s: s.first_valid_index()),
        "last_observation": stock_px.apply(lambda s: s.last_valid_index()),
    })
    return stock_px, benchmark_px, coverage

In [ ]:
raw = fetch_bloomberg_prices(CFG)
display(raw.head())
print("返回列：", list(raw.columns))
prices, benchmark_px, coverage = prepare_price_panel(raw, CFG)
display(coverage)
print("观测区间：", prices.index.min(), "至", prices.index.max())
print("价格矩阵：", prices.shape)

In [ ]:
# 仅用于人工检查异常，不自动删去大幅波动。
daily_check = prices.pct_change(fill_method=None)
large_moves = daily_check.where(daily_check.abs() > 0.25).stack().rename("daily_change")
print("绝对日变化超过 25% 的观测（检查公司行动或字段调整，不自动视为错误）：")
display(large_moves.to_frame().head(30))

## 3｜构建因子与区间末选股快照

`pct_change(fill_method=None)` 返回小数收益，例如 `0.02` 表示 2%，不是“2”。
动量用正的 `shift` 取过去价格；从不将未来数据回填到信号。
每个截面只对同时具有有效动量和波动率的股票排名，并检查截至该日的完整历史。

排名分数范围为 `(0, 1]`，越高越优。代码只用已有历史决定当时是否纳入，
不会因一只股票在更晚时期缺数据而删除其早期记录。该规则不等于消除了固定池的生存偏差。

In [ ]:
def build_factors(prices: pd.DataFrame, cfg: Config) -> Dict[str, pd.DataFrame]:
    """12-1 月动量以 252/21 个观测日近似；低波动以 63 日标准差定义。"""
    if not prices.index.is_monotonic_increasing or prices.index.has_duplicates:
        raise ValueError("价格索引必须按时间升序且不能重复。")
    returns = prices.pct_change(fill_method=None)
    momentum = prices.shift(cfg.momentum_skip) / prices.shift(cfg.momentum_lookback) - 1
    volatility = returns.rolling(
        cfg.volatility_window, min_periods=cfg.volatility_window
    ).std(ddof=1) * np.sqrt(cfg.trading_days)
    # 每个时点仅检查截至该时点的历史，不能按未来数据完整性筛选。
    history_ok = prices.notna().rolling(
        cfg.momentum_lookback + 1, min_periods=cfg.momentum_lookback + 1
    ).sum().eq(cfg.momentum_lookback + 1)
    eligible = history_ok & momentum.notna() & volatility.notna() & prices.notna()
    mom_rank = momentum.where(eligible).rank(axis=1, pct=True)
    lowvol_rank = (-volatility).where(eligible).rank(axis=1, pct=True)
    score = cfg.momentum_weight * mom_rank + (1 - cfg.momentum_weight) * lowvol_rank
    score = score.where(eligible.sum(axis=1).ge(cfg.min_names), axis=0)
    return {"returns": returns, "momentum": momentum, "volatility": volatility,
            "mom_rank": mom_rank, "lowvol_rank": lowvol_rank, "score": score}


def latest_ranking(factors: Dict[str, pd.DataFrame], cfg: Config) -> pd.DataFrame:
    """区间最后一个观测日的选股快照，不是当时实际持仓或买入建议。"""
    last = factors["score"].index[-1]
    table = pd.DataFrame({
        "momentum_12_1": factors["momentum"].loc[last],
        "volatility_63d_ann": factors["volatility"].loc[last],
        "momentum_percentile": factors["mom_rank"].loc[last],
        "lowvol_percentile": factors["lowvol_rank"].loc[last],
        "score": factors["score"].loc[last],
    }).dropna(subset=["score"])
    table = table.sort_index().sort_values("score", ascending=False, kind="mergesort")
    table["illustrative_target_weight"] = 0.0
    if len(table) >= cfg.min_names:
        table.loc[table.index[:cfg.top_n], "illustrative_target_weight"] = 1 / cfg.top_n
    table.attrs["as_of"] = str(last.date())
    return table

In [ ]:
factors = build_factors(prices, CFG)
ranking = latest_ranking(factors, CFG)
print("选股快照日期：", ranking.attrs["as_of"])
display(ranking.style.format({
    "momentum_12_1": "{:.2%}", "volatility_63d_ann": "{:.2%}",
    "momentum_percentile": "{:.3f}", "lowvol_percentile": "{:.3f}",
    "score": "{:.3f}", "illustrative_target_weight": "{:.2%}"
}))

## 4｜月度组合与成本记账

**时间轴：**月末 $t$ 收盘得分 → $t+1$ 收盘成交 → 新组合从 $t+1$ 收盘后开始承担收益。
执行当天先计算旧持仓的收盘到收盘收益，再扣交易成本并调仓。

交易成本按实际漂移后的持仓与新目标计算。若交易前净值归一化为 1，
扣费后净值为 $z$，目标为 $w^*$，交易前权重为 $w^-$，则解：

$$z+c\sum_i|zw_i^*-w_i^-|=1.$$

`traded_notional_fraction` 是买卖金额绝对值之和除以交易前净值，**没有除以 2**。
包含初始建仓成本，但样本末仅估值、不强制清仓，因此没有末端清仓成本。
假设可在收盘价成交、可交易小数份额，没有市场冲击、最低佣金、税费和成交量限制。
持仓收益或待交易价格缺失即停止，避免把缺失当作零收益。

In [ ]:
def monthly_targets(score: pd.DataFrame, cfg: Config):
    """月末收盘评分；下一基准观测日收盘成交。最后未完成月份不强行调仓。"""
    dates = score.index
    month = dates.to_period("M")
    signal_positions = np.flatnonzero(month[:-1] != month[1:])
    targets, log = {}, []
    for pos in signal_positions:
        ranked = score.iloc[pos].dropna().sort_index()
        ranked = ranked.sort_values(ascending=False, kind="mergesort")
        if len(ranked) < cfg.min_names:
            if targets:
                raise ValueError("%s 可评分股票不足；请明确缺失/停牌政策。" % dates[pos])
            continue  # 起始阶段仅为预热
        selected = ranked.head(cfg.top_n).index
        target = pd.Series(0.0, index=score.columns)
        target.loc[selected] = 1 / cfg.top_n
        execution = dates[pos + 1]
        targets[execution] = target
        log.append({"signal_date": dates[pos], "execution_date": execution,
                    "eligible_names": len(ranked), "selected": "; ".join(selected)})
    if not targets:
        raise ValueError("没有可执行的月度信号；请扩大历史区间或核查股票覆盖。")
    out = pd.DataFrame.from_dict(targets, orient="index").sort_index()
    out.index.name = "execution_date"
    return out, pd.DataFrame(log)


def solve_cost_fraction(pre_weights: np.ndarray, target: np.ndarray, rate: float):
    """精确解简单比例成本模型：z + c*sum(abs(z*target-pre_weights)) = 1。

    z 为交易后/交易前净值；交易金额按买、卖两侧名义金额分别计费。
    这不是冲击模型，不包含最小佣金、整股交易、税费或借券成本。
    """
    if not 0 <= rate < 1:
        raise ValueError("比例成本必须在 [0,1) 内。")
    if rate == 0:
        return 1.0, float(np.abs(target - pre_weights).sum())
    lower, upper = 0.0, 1.0
    for _ in range(60):
        z = (lower + upper) / 2
        error = z + rate * np.abs(z * target - pre_weights).sum() - 1
        if error > 0:
            upper = z
        else:
            lower = z
    z = (lower + upper) / 2
    return z, float(np.abs(z * target - pre_weights).sum())


def backtest_monthly(prices: pd.DataFrame, targets: pd.DataFrame, cost_bps: float):
    """多头、允许小数份额、现金收益为零；非调仓日让权重自然漂移。

    当日先计算旧持仓的收盘到收盘收益，再于收盘成交。
    所以新组合不会获得信号日或成交前的收益。持仓缺失数据立即报错。
    """
    if targets.empty or targets.index.has_duplicates:
        raise ValueError("targets 不能为空或包含重复执行日期。")
    if not targets.index.isin(prices.index).all():
        raise ValueError("执行日期不在价格索引中。")
    targets = targets.reindex(columns=prices.columns)
    tv = targets.to_numpy(dtype=float)
    if not np.isfinite(tv).all() or (tv < 0).any():
        raise ValueError("本引擎仅支持有限、非负目标权重。")
    if (targets.sum(axis=1) > 1 + 1e-10).any():
        raise ValueError("目标权重不能超过 100%；本例不允许杠杆。")
    rate = cost_bps / 10000
    if not np.isfinite(rate) or not 0 <= rate < 1:
        raise ValueError("cost_bps 无效。")
    daily = prices.pct_change(fill_method=None)
    w = pd.Series(0.0, index=prices.columns)
    rows, weight_rows = [], []
    for date in prices.loc[targets.index.min():].index:
        ri = daily.loc[date]
        held = w > 0
        if ri.loc[held].isna().any() or not np.isfinite(ri.loc[held]).all():
            bad = list(ri.loc[held][~np.isfinite(ri.loc[held])].index)
            raise ValueError("%s 持仓收益缺失: %s；不可填零或静默删除。" % (date, bad))
        if (ri.loc[held] <= -1).any():
            raise ValueError("出现 -100% 或更低收益，须单独处理退市/清算。")
        gross = float((w.loc[held] * ri.loc[held]).sum())
        assets = pd.Series(0.0, index=prices.columns)
        assets.loc[held] = w.loc[held] * (1 + ri.loc[held])
        pre = assets / (1 + gross)  # 收盘交易之前的漂移权重
        z, traded = 1.0, 0.0
        if date in targets.index:
            target = targets.loc[date]
            touched = (pre > 0) | (target > 0)
            if prices.loc[date, touched].isna().any():
                raise ValueError("%s 待交易证券缺价；应显式处理无法成交。" % date)
            z, traded = solve_cost_fraction(pre.to_numpy(), target.to_numpy(), rate)
            w = target.copy()  # 成本已支付，目标按扣费后净值归一化
        else:
            w = pre
        rows.append({"date": date, "gross_return": gross,
                     "net_return": (1 + gross) * z - 1,
                     "traded_notional_fraction": traded,
                     "cost_fraction_pretrade_nav": 1 - z,
                     "holdings": int((w > 0).sum()),
                     "cash_weight": max(0.0, 1 - float(w.sum()))})
        weight_rows.append(w.rename(date))
    return pd.DataFrame(rows).set_index("date"), pd.DataFrame(weight_rows)

## 5｜绩效、月度 Rank IC 与图表

CAGR 按首个执行日到最后观测日的实际日历跨度计算。波动率和 Sharpe 使用 252 日年化；
Sharpe 的无风险利率为参数假设。最大回撤包含交易前初始净值 1。
比较基准也从首个执行日收盘开始，故首日不计入此前的基准涨跌。

Rank IC 将月末评分与“下一执行日收盘至再下一执行日收盘”的收益排名相关。
月份之间不重叠，不将未来收益用于构建信号；退出价格缺失时拒绝静默剔除。
只提供描述统计，不根据这个小样本报告显著性或可投资性结论。

In [ ]:
def performance_table(return_panel: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """含初始建仓成本；CAGR 按实际日历跨度，波动率/Sharpe 按 252 日年化。"""
    if len(return_panel) < 2 or return_panel.isna().any().any():
        raise ValueError("绩效表至少需要两个无缺失观测。")
    if (return_panel <= -1).any().any():
        raise ValueError("本绩效函数不处理净值归零或负值。")
    years = (return_panel.index[-1] - return_panel.index[0]).days / 365.25
    if years <= 0:
        raise ValueError("有效日历跨度必须为正。")
    rf_daily = (1 + cfg.risk_free_annual) ** (1 / cfg.trading_days) - 1
    rows = {}
    for name, r in return_panel.items():
        nav = (1 + r).cumprod()
        drawdown = nav / nav.cummax().clip(lower=1.0) - 1  # 包含初始净值 1
        sd = r.std(ddof=1)
        rows[name] = {
            "cumulative_return": nav.iloc[-1] - 1,
            "CAGR": nav.iloc[-1] ** (1 / years) - 1,
            "annualized_volatility": sd * np.sqrt(cfg.trading_days),
            "Sharpe_assumed_rf": ((r.mean() - rf_daily) / sd * np.sqrt(cfg.trading_days)
                                  if sd > 0 else np.nan),
            "max_drawdown": drawdown.min(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


def monthly_rank_ic(prices: pd.DataFrame, score: pd.DataFrame,
                    decisions: pd.DataFrame) -> pd.DataFrame:
    """月末评分 vs 下一执行日收盘到再下一执行日收盘的收益；不重叠。

    未来收益仅用于事后检验，绝不用于构建 score。缺失退出价格时不静默剔除。
    本例为小样本描述统计，不报告显著性或策略有效性结论。
    """
    rows = []
    for k in range(len(decisions) - 1):
        current, following = decisions.iloc[k], decisions.iloc[k + 1]
        start, end = current.execution_date, following.execution_date
        s = score.loc[current.signal_date].dropna()
        fwd = prices.loc[end, s.index] / prices.loc[start, s.index] - 1
        if not np.isfinite(fwd).all():
            raise ValueError("IC 的未来收益缺失；先处理退市/停牌，不能只保留幸存证券。")
        value = s.rank().corr(fwd.rank()) if s.nunique() > 1 and fwd.nunique() > 1 else np.nan
        rows.append({"signal_date": current.signal_date, "execution_from": start,
                     "execution_to": end, "names": len(s), "rank_ic": value})
    if not rows:
        return pd.DataFrame(columns=["execution_from", "execution_to", "names", "rank_ic"])
    return pd.DataFrame(rows).set_index("signal_date")


def analyze(prices: pd.DataFrame, benchmark_px: pd.Series, cfg: Config):
    """纯 Python 分析入口；不连接 Bloomberg，也不保存或外传数据。"""
    if not prices.index.equals(benchmark_px.index):
        raise ValueError("股票与基准必须使用相同日期索引。")
    factors = build_factors(prices, cfg)
    targets, decisions = monthly_targets(factors["score"], cfg)
    ledger, weights = backtest_monthly(prices, targets, cfg.cost_bps)
    benchmark_returns = benchmark_px.pct_change(fill_method=None).reindex(ledger.index)
    benchmark_returns.iloc[0] = 0.0  # 首次执行日收盘才开始比较，不偷取当日收益
    panel = pd.DataFrame({"strategy_gross": ledger.gross_return,
                          "strategy_net": ledger.net_return,
                          "benchmark_no_cost": benchmark_returns})
    return {"factors": factors, "ranking": latest_ranking(factors, cfg),
            "targets": targets, "decisions": decisions, "ledger": ledger,
            "weights": weights, "returns": panel,
            "performance": performance_table(panel, cfg),
            "ic": monthly_rank_ic(prices, factors["score"], decisions)}


def plot_results(result, cfg: Config):
    """每幅图单独绘制；标题保留未经确认的收益口径标识。"""
    nav = (1 + result["returns"]).cumprod()
    # 在首个执行日期补上交易前净值 1，图形也包含最初建仓成本。
    initial = pd.DataFrame(1.0, index=[nav.index[0]], columns=nav.columns)
    plotted = pd.concat([initial, nav])
    plt.figure(figsize=(10, 4))
    for column in plotted:
        plt.plot(plotted.index, plotted[column], label=column)
    plt.title(cfg.series_label + " | NAV")
    plt.ylabel("Net asset value")
    plt.legend()
    plt.tight_layout()
    plt.show()

    drawdown = nav / nav.cummax().clip(lower=1.0) - 1
    plt.figure(figsize=(10, 3.5))
    plt.plot(drawdown.index, drawdown["strategy_net"])
    plt.title(cfg.series_label + " | Drawdown")
    plt.ylabel("Drawdown (fraction)")
    plt.tight_layout()
    plt.show()

    ic = result["ic"]
    if not ic.empty:
        plt.figure(figsize=(10, 3.5))
        plt.bar(ic.index, ic["rank_ic"], width=15)
        plt.axhline(0, linewidth=0.8)
        plt.title("Monthly rank IC - descriptive, fixed sample")
        plt.ylabel("Rank IC")
        plt.tight_layout()
        plt.show()

In [ ]:
result = analyze(prices, benchmark_px, CFG)
print(CFG.series_label)
display(result["performance"].style.format({
    "cumulative_return": "{:.2%}", "CAGR": "{:.2%}",
    "annualized_volatility": "{:.2%}", "Sharpe_assumed_rf": "{:.3f}",
    "max_drawdown": "{:.2%}"
}))
print("信号与执行日审计：")
display(result["decisions"].head(8))
print("最近执行后的实际漂移权重，不同于区间末选股快照：")
display(result["weights"].tail())

In [ ]:
print("月度交易金额与成本：")
trade_days = result["ledger"].loc[result["targets"].index]
display(trade_days[["traded_notional_fraction", "cost_fraction_pretrade_nav", "holdings"]].head(12))
print("Rank IC：")
display(result["ic"])
if not result["ic"].empty:
    display(result["ic"]["rank_ic"].describe().to_frame("descriptive_only"))

In [ ]:
# 运行时断言：辅助检查；不证明经济有效性。
assert np.allclose(result["targets"].sum(axis=1), 1.0)
assert np.allclose(result["weights"].sum(axis=1), 1.0)
assert not result["returns"].isna().any().any()
assert result["returns"]["benchmark_no_cost"].iloc[0] == 0.0
for row in result["decisions"].itertuples():
    assert prices.index.get_loc(row.execution_date) == prices.index.get_loc(row.signal_date) + 1
print("运行时检查通过。请仍核查价格口径、历史股票池和可交易性。")

In [ ]:
plot_results(result, CFG)

## 6｜可选基本面扩展：仅最新快照，默认关闭

候选映射 `pe_ratio`、`return_com_eqy`、`cur_mkt_cap` **未在用户环境验证**。
逐字段返回状态与原始异常，不隐藏失败。请让培训师确认财务期间、单位、币种、
报告可用日期以及权限。仅将 `PE > 0` 转成 earnings yield 是演示选择，不是适合所有行业的规则。

当前基本面不能与默认的 **2025-12-31** 价格快照拼接。
四因子函数要求经确认的 `fundamentals.attrs["verified_as_of"]` 与价格快照日期相同，
且只用于该时点截面。不要仅为了绕过检查而填写日期。

历史多因子研究还需要逐时点可用的财务值、公告/可用时间、财年和重述政策；
本模板没有实现或声称具备这样的历史基本面回测。

In [ ]:
def optional_fundamental_snapshot(tickers):
    """候选字段由培训师验证；逐字段报告错误，不用 NaN 掩盖请求失败。

    pe_ratio / return_com_eqy / cur_mkt_cap 为待现场确认的字段映射。
    默认参数的会计期间、单位、币种及报告日期均不可视为已确认。
    """
    import bql
    bq = bql.Service()
    candidates = {"pe": "pe_ratio", "roe_raw": "return_com_eqy",
                  "market_cap_raw": "cur_mkt_cap"}
    frame = pd.DataFrame(index=pd.Index(list(tickers), name="ID"))
    status = []
    for alias, field in candidates.items():
        try:
            request = bql.Request(list(tickers), {alias: getattr(bq.data, field)()})
            value = bq.execute(request)[0].df()
            if value.index.has_duplicates or alias not in value.columns:
                raise ValueError("返回维度不是每只股票一行，请明确财务期间。")
            series = pd.to_numeric(value[alias], errors="raise")
            if len(set(series.index).intersection(frame.index)) == 0:
                raise ValueError("返回证券 ID 与输入不一致，请检查映射。")
            frame[alias] = series.reindex(frame.index)
            status.append({"field": field, "status": "OK - semantics need review", "error": ""})
        except Exception as exc:
            status.append({"field": field, "status": "FAILED", "error": str(exc)})
    return frame, pd.DataFrame(status)


def current_four_factor_ranking(price_ranking: pd.DataFrame,
                                fundamentals: pd.DataFrame) -> pd.DataFrame:
    """仅适用于同一当前时点的演示；不能与历史截止日价格快照混合。"""
    verified_as_of = fundamentals.attrs.get("verified_as_of")
    if verified_as_of is None or verified_as_of != price_ranking.attrs.get("as_of"):
        raise ValueError(
            "基本面与价格日期未确认一致。须先由培训师确认数据时点，"
            "再设置 fundamentals.attrs['verified_as_of']；不能拼接当前基本面与历史价格。"
        )
    if not {"pe", "roe_raw"}.issubset(fundamentals.columns):
        raise ValueError("PE/ROE 不完整；请先修正字段请求，不要用零填充。")
    table = price_ranking.join(fundamentals)
    table["earnings_yield"] = 1 / table["pe"].where(table["pe"] > 0)
    complete = table.dropna(subset=["momentum_12_1", "volatility_63d_ann",
                                   "earnings_yield", "roe_raw"]).copy()
    # 所有分项在同一个有效股票子集内排名。
    complete["four_factor_score"] = (
        complete["momentum_12_1"].rank(pct=True)
        + (-complete["volatility_63d_ann"]).rank(pct=True)
        + complete["earnings_yield"].rank(pct=True)
        + complete["roe_raw"].rank(pct=True)
    ) / 4
    return complete.sort_values("four_factor_score", ascending=False)

In [ ]:
RUN_OPTIONAL_FUNDAMENTALS = False
if RUN_OPTIONAL_FUNDAMENTALS:
    fundamentals, field_status = optional_fundamental_snapshot(CFG.tickers)
    display(field_status)
    display(fundamentals)
    # 不自动合并到历史价格排名，不输入历史回测。
else:
    print("基本面扩展未运行。请先与培训师确认字段与时点。")

## 7｜带到培训现场的核查表

| 请培训师现场演示 | 对本代码的影响 |
|---|---|
| 确认价格与全收益字段、拆股及分红调整 | 替换查询层并标记一致的收益口径 |
| 获取历史时点股票池，包含退市和旧证券标识 | 替换固定股票池；建立逐日 eligibility |
| 明确停牌、退市和缺价时的估值/成交政策 | 替换当前“缺失即报错”的简化规则 |
| 获取 point-in-time 财务与预期，说明公告可用日期 | 扩展价值/质量/预期修正，不使用当前快照回填 |
| 加入行业和规模暴露约束、风险模型 | 当前排名不是行业中性或风险中性组合 |
| 核查查询规模、包版本、数据保存和团队共享权限 | 在获授权的范围内扩展与复现 |

交付前本地验证：13 组纯 Python 模拟数据测试，另含 100 个随机组合的成本恒等式检查。
测试覆盖成交时点、自然权重漂移、历史信号不受未来价格修改影响、缺失数据报错、
初始资金回撤基线与 IC 退出价格检查。测试环境 numpy 2.3.5 / pandas 2.2.3；
并未据此验证 Bloomberg 专有客户端或用户的数据权限。

## 参考资料与 API 边界

- [Bloomberg：BQuant 产品与数据权限概览](https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/)
- [Bloomberg：Point-in-Time Financials / Estimates / Pricing 产品说明](https://professional.bloomberg.com/products/data/enterprise-catalog/cofi/)
- [pandas：DataFrame.pct_change](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html)
- [pandas：DataFrame.shift](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html)
- [Kenneth French：Monthly Momentum Factor 定义](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_mom_factor.html)
- [Mingze Gao：本人 BQuant 示例中的 Service / Request / execute 调用](https://mingze-gao.com/posts/bloomberg-bquant/)

Bloomberg 产品介绍并不是完整的 Python API 文档。公开示例只用于核对调用结构；
专有字段、参数和默认语义以当前 BQuant 内帮助与现场执行结果为准。
代码为本次任务编写，未复制完整第三方策略，也未验证或推荐任何投资结果。